# TMDB Movie Dataset Pipeline

Builds a movie dataset from TMDB (2005–2024) with posters, scripts, and inflation-adjusted ROI.

**Pipeline stages:**
1. Discover movie IDs from TMDB Discover API
2. Fetch full metadata per movie (budget, gross, genres, certification, etc.)
3. Download poster images; generate synthetic posters for missing ones
4. Scrape movie scripts from IMSDb and fallback sources
5. Enrich missing ROI from Box Office Mojo
6. Produce clean CSVs with 70/30 year-stratified splits

**Notes:**
- Set `TMDB_BEARER_TOKEN` in your environment or Colab secrets before running.
- Upload `absent_poster.jpg` to `BASE_DIR` before running synthetic poster generation.
- ROI is inflation-adjusted to 2025 USD using CPI data.
- pandas reads missing CSV values as `float NaN`, not `None`; use `_is_nan()` throughout.

## Installation

In [ ]:
%pip install requests pillow pandas numpy scikit-learn beautifulsoup4

## Imports

In [ ]:
import os
import re
import time
import unicodedata
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split

## Configuration

In [ ]:
# Set TMDB_BEARER_TOKEN in your environment or Colab secrets.
BEARER_TOKEN = os.environ.get("TMDB_BEARER_TOKEN", "")
HEADERS      = {"Authorization": f"Bearer {BEARER_TOKEN}"}

TMDB_BASE = "https://api.themoviedb.org/3"
IMG_BASE  = "https://image.tmdb.org/t/p/w500"

IMSDB_BASE   = "https://imsdb.com"
SIMPLY_BASE  = "https://www.simplyscripts.com"
SLUG_BASE    = "https://www.scriptslug.com"
ASTREET_BASE = "https://search.alexanderstreet.com"
BOM_BASE     = "https://www.boxofficemojo.com"

START_YEAR       = 2005
END_YEAR         = 2024
CHECKPOINT_EVERY = 1000

TMDB_WORKERS   = 10
POSTER_WORKERS = 10
IMSDB_WORKERS  = 5

BASE_DIR   = Path("/content/gdrive/Shareddrives/FML_FINAL/Data")
POSTER_DIR = BASE_DIR / "posters"
SCRIPT_DIR = BASE_DIR / "scripts"
CKPT_DIR   = BASE_DIR / "checkpoints"

# absent_poster.jpg must be uploaded to BASE_DIR before synthetic generation.
ABSENT_POSTER_PATH     = BASE_DIR / "absent_poster.jpg"
SYNTHETIC_POSTER_NOISE = 15   # Gaussian noise std (0-255 scale)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
BOM_HEADERS = {
    **REQUEST_HEADERS,
    "Accept-Language": "en-US,en;q=0.9",
    "Referer":         "https://www.boxofficemojo.com/",
}

for _d in [BASE_DIR, POSTER_DIR, SCRIPT_DIR, CKPT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

## Utility Helpers

- `_is_nan` / `_file_ok`: NaN-safe guards used throughout in place of bare `is not None` checks.
- `adjust_to_2025` / `compute_roi`: inflation adjustment using BLS CPI data.

In [ ]:
CPI = {
    2003: 184.0, 2004: 188.9, 2005: 195.3, 2006: 201.6, 2007: 207.3,
    2008: 215.3, 2009: 214.5, 2010: 218.1, 2011: 224.9, 2012: 229.6,
    2013: 233.0, 2014: 236.7, 2015: 237.0, 2016: 240.0, 2017: 245.1,
    2018: 251.1, 2019: 255.7, 2020: 258.8, 2021: 270.9, 2022: 292.7,
    2023: 304.7, 2024: 314.2, 2025: 319.8,
}
CPI_2025 = CPI[2025]


def _is_nan(val) -> bool:
    """True if val is None or float NaN."""
    if val is None:
        return True
    return isinstance(val, float) and pd.isna(val)


def _file_ok(val) -> bool:
    """True if val is a non-empty string pointing to an existing file."""
    return isinstance(val, str) and bool(val) and Path(val).exists()


def adjust_to_2025(value: float, year: int) -> "float | None":
    if year not in CPI or CPI[year] == 0:
        return None
    return value * (CPI_2025 / CPI[year])


def compute_roi(gross: float, budget: float, release_year: int):
    """Returns (adj_gross, adj_budget, roi) in 2025 USD, or (None, None, None)."""
    if not gross or not budget or gross <= 0 or budget <= 0:
        return None, None, None
    adj_gross  = adjust_to_2025(gross,  release_year)
    adj_budget = adjust_to_2025(budget, release_year - 1)
    if adj_gross is None or adj_budget is None or adj_budget == 0:
        return None, None, None
    roi = (adj_gross - adj_budget) / adj_budget
    return round(adj_gross, 2), round(adj_budget, 2), round(roi, 6)

## TMDB API

In [ ]:
def discover_movie_ids(year: int) -> list:
    """Returns all US theatrical movie IDs for a given year via TMDB Discover."""
    ids, page = [], 1
    while True:
        data = None
        for attempt in range(3):
            try:
                r = requests.get(
                    f"{TMDB_BASE}/discover/movie",
                    headers=HEADERS,
                    params={
                        "region":            "US",
                        "release_date.gte":  f"{year}-01-01",
                        "release_date.lte":  f"{year}-12-31",
                        "with_release_type": "3",
                        "sort_by":           "popularity.desc",
                        "page":              page,
                    },
                    timeout=15,
                )
                r.raise_for_status()
                data = r.json()
                break
            except requests.exceptions.HTTPError:
                status = r.status_code if r is not None else 0
                if status in (500, 429):
                    time.sleep(10 if status == 429 else 5)
                    if attempt == 2:
                        data = {"results": [], "total_pages": page}
                else:
                    raise
            except requests.exceptions.RequestException:
                time.sleep(3)
                if attempt == 2:
                    data = {"results": [], "total_pages": page}
        if data is None:
            break
        ids.extend(m["id"] for m in data.get("results", []))
        total_pages = data.get("total_pages", 1)
        if page % 50 == 0:
            print(f"      page {page}/{total_pages}  ({len(ids):,} IDs so far)")
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.05)
    return ids


def get_movie_details(tmdb_id: int) -> "dict | None":
    """Fetches full movie details including release_dates from TMDB."""
    for attempt in range(2):
        try:
            r = requests.get(
                f"{TMDB_BASE}/movie/{tmdb_id}",
                headers=HEADERS,
                params={"append_to_response": "release_dates"},
                timeout=15,
            )
            if r.status_code == 429:
                time.sleep(10)
                continue
            return r.json() if r.status_code == 200 else None
        except requests.exceptions.RequestException:
            time.sleep(2)
    return None


def get_us_certification(release_dates_data: dict) -> "str | None":
    """Extracts the US MPAA rating from TMDB release_dates response."""
    for entry in release_dates_data.get("results", []):
        if entry["iso_3166_1"] == "US":
            for rd in entry.get("release_dates", []):
                cert = rd.get("certification", "").strip()
                if cert:
                    return cert
    return None


def search_tmdb_by_title(title: str, year: int = None) -> "int | None":
    """
    Searches TMDB for a movie by title, optionally filtered by year.
    Prefers exact title match; breaks ties by highest popularity.
    """
    params = {"query": title, "include_adult": False}
    if year and not _is_nan(year):
        params["year"] = int(year)
    for attempt in range(3):
        try:
            r = requests.get(f"{TMDB_BASE}/search/movie",
                             headers=HEADERS, params=params, timeout=15)
            if r.status_code == 429:
                time.sleep(10)
                continue
            r.raise_for_status()
            results = r.json().get("results", [])
            if not results:
                return None
            title_lower = title.lower().strip()
            exact = [m for m in results if m.get("title", "").lower().strip() == title_lower]
            pool  = exact if exact else results
            return max(pool, key=lambda m: m.get("popularity", 0))["id"]
        except requests.exceptions.RequestException:
            time.sleep(2)
    return None

## Poster Handling

`generate_synthetic_poster` creates a reproducibly-noisy copy of `absent_poster.jpg`
for movies with no real poster. Seed = `tmdb_id % 2^32` gives each movie a unique but
deterministic noise pattern.

In [ ]:
def download_poster(tmdb_id: int, poster_path: str) -> "str | None":
    """Downloads a poster from TMDB and saves it to POSTER_DIR/{tmdb_id}.jpg."""
    if not poster_path or _is_nan(poster_path):
        return None
    save_path = POSTER_DIR / f"{tmdb_id}.jpg"
    if save_path.exists():
        return str(save_path)
    try:
        r = requests.get(IMG_BASE + poster_path, timeout=15)
        r.raise_for_status()
        Image.open(BytesIO(r.content)).convert("RGB").save(save_path)
        return str(save_path)
    except Exception as e:
        print(f"    x Poster failed [{tmdb_id}]: {e}")
        return None


def generate_synthetic_poster(tmdb_id: int) -> "str | None":
    """
    Creates a uniquely-noisy copy of absent_poster.jpg for movies without a real poster.
    Skips if any poster already exists on disk. Returns the local path or None.
    """
    save_path = POSTER_DIR / f"{tmdb_id}.jpg"
    if save_path.exists():
        return str(save_path)
    if not ABSENT_POSTER_PATH.exists():
        return None
    try:
        img   = Image.open(ABSENT_POSTER_PATH).convert("RGB")
        arr   = np.array(img, dtype=np.float32)
        rng   = np.random.default_rng(seed=int(tmdb_id) % (2**32))
        noise = rng.normal(0, SYNTHETIC_POSTER_NOISE, arr.shape)
        arr   = np.clip(arr + noise, 0, 255).astype(np.uint8)
        Image.fromarray(arr).save(save_path, quality=90)
        return str(save_path)
    except Exception as e:
        print(f"    x Synthetic poster failed [{tmdb_id}]: {e}")
        return None


def _synthetic_poster_worker(record: dict) -> dict:
    if _file_ok(record.get("poster_file")):
        return record
    record["poster_file"] = generate_synthetic_poster(record["tmdb_id"])
    return record


def fill_missing_posters(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates synthetic posters for every movie that lacks a real one on disk.
    Requires absent_poster.jpg in BASE_DIR. Saves updated movies_raw.csv.
    """
    if not ABSENT_POSTER_PATH.exists():
        print(f"  [ERROR] {ABSENT_POSTER_PATH} not found — upload absent_poster.jpg and re-run.")
        return df
    missing_mask    = ~df["poster_file"].apply(_file_ok)
    missing_records = df[missing_mask].to_dict("records")
    ok_records      = df[~missing_mask].to_dict("records")
    print(f"  Real posters on disk:      {len(ok_records):,}")
    print(f"  Missing (will synthesise): {len(missing_records):,}")
    if not missing_records:
        return df
    generated = []
    success = fail = 0
    with ThreadPoolExecutor(max_workers=POSTER_WORKERS) as executor:
        futures = {executor.submit(_synthetic_poster_worker, rec): rec for rec in missing_records}
        for future in as_completed(futures):
            result  = future.result()
            generated.append(result)
            ok       = _file_ok(result.get("poster_file"))
            success += ok
            fail    += not ok
            done     = success + fail
            if done % 1000 == 0:
                pct = done / len(missing_records) * 100
                print(f"  Synthetic: {done:,}/{len(missing_records):,} ({pct:.1f}%)  "
                      f"|  created: {success:,}  |  failed: {fail:,}")
    print(f"\n  Synthetic posters created: {success:,}  |  Failed: {fail:,}")
    df_updated = pd.DataFrame(ok_records + generated)
    df_updated.to_csv(BASE_DIR / "movies_raw.csv", index=False)
    print(f"  Updated movies_raw.csv ({len(df_updated):,} rows)")
    return df_updated

## Script Scraping

Sources tried in order per movie: **IMSDb → SimplyScripts → ScriptSlug → Alexander Street**.
Script-O-Rama is excluded (returns 403).

In [ ]:
def slugify(title: str) -> str:
    """Converts a movie title to a URL-safe hyphenated slug."""
    title = unicodedata.normalize("NFKD", title).encode("ascii", "ignore").decode("ascii")
    title = re.sub(r"[^\w\s-]", "", title)
    return re.sub(r"\s+", "-", title.strip())


# ── IMSDb ──────────────────────────────────────────────────────────────────

def _parse_imsdb_script_page(html: str) -> "str | None":
    soup = BeautifulSoup(html, "html.parser")
    scrtext = soup.find("td", class_="scrtext")
    if scrtext:
        pre = scrtext.find("pre")
        if pre:
            return pre.get_text()
    for pre in soup.find_all("pre"):
        text = pre.get_text()
        if len(text) > 5000:
            return text
    return None


def _parse_imsdb_search_results(html: str, title: str) -> "str | None":
    soup        = BeautifulSoup(html, "html.parser")
    title_lower = title.lower().strip()
    for a in soup.find_all("a", href=True):
        href      = a["href"]
        link_text = a.get_text(strip=True).lower()
        if "/Movie Scripts/" not in href and "script" not in href.lower():
            continue
        clean = link_text.replace(" script", "").strip()
        if clean == title_lower or title_lower in clean or clean in title_lower:
            return IMSDB_BASE + href
    return None


def fetch_imsdb_script(title: str, tmdb_id: int) -> "str | None":
    """Fetches a script from IMSDb by direct slug URL, then search fallback."""
    save_path = SCRIPT_DIR / f"{tmdb_id}.txt"
    if save_path.exists():
        return str(save_path)
    slug        = slugify(title)
    script_text = None
    try:
        r = requests.get(f"{IMSDB_BASE}/scripts/{slug}.html",
                         headers=REQUEST_HEADERS, timeout=15)
        if r.status_code == 200:
            script_text = _parse_imsdb_script_page(r.text)
    except Exception:
        pass
    time.sleep(0.5)
    if not script_text:
        try:
            r = requests.get(f"{IMSDB_BASE}/search.php",
                             headers=REQUEST_HEADERS, params={"query": title}, timeout=15)
            if r.status_code == 200:
                url = _parse_imsdb_search_results(r.text, title)
                if url:
                    time.sleep(0.5)
                    r2 = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
                    if r2.status_code == 200:
                        script_text = _parse_imsdb_script_page(r2.text)
        except Exception:
            pass
        time.sleep(0.5)
    if script_text and len(script_text.strip()) > 500:
        save_path.write_text(script_text, encoding="utf-8")
        return str(save_path)
    return None


# ── SimplyScripts ──────────────────────────────────────────────────────────

def _parse_simply_page(html: str) -> "str | None":
    soup = BeautifulSoup(html, "html.parser")
    for pre in soup.find_all("pre"):
        if len(pre.get_text()) > 2000:
            return pre.get_text()
    for div in soup.find_all("div"):
        if len(div.get_text()) > 8000:
            return div.get_text()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".txt"):
            url = href if href.startswith("http") else SIMPLY_BASE + href
            try:
                r = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
                if r.status_code == 200 and len(r.text) > 500:
                    return r.text
            except Exception:
                pass
    return None


def _parse_simply_search(html: str, title: str) -> "str | None":
    soup        = BeautifulSoup(html, "html.parser")
    title_lower = title.lower().strip()
    for a in soup.find_all("a", href=True):
        href      = a["href"]
        link_text = a.get_text(strip=True).lower()
        if not any(x in href.lower() for x in ["/scripts/", "script", "/movie"]):
            continue
        clean = re.sub(r"\b(script|screenplay|the)\b", "", link_text, flags=re.IGNORECASE).strip()
        if clean == title_lower or title_lower in clean or clean in title_lower:
            return href if href.startswith("http") else SIMPLY_BASE + href
    return None


def fetch_simplyscripts(title: str, tmdb_id: int) -> "str | None":
    """Fetches a script from SimplyScripts by direct URL patterns, then search fallback."""
    save_path = SCRIPT_DIR / f"{tmdb_id}.txt"
    if save_path.exists():
        return str(save_path)
    slug = slugify(title)
    for url in [
        f"{SIMPLY_BASE}/scripts/{slug}.html",
        f"{SIMPLY_BASE}/Movie_Scripts/{slug}-Script.html",
        f"{SIMPLY_BASE}/unproduced/{slug}.html",
    ]:
        try:
            r = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
            if r.status_code == 200:
                text = _parse_simply_page(r.text)
                if text and len(text.strip()) > 500:
                    save_path.write_text(text, encoding="utf-8")
                    return str(save_path)
        except Exception:
            pass
        time.sleep(0.3)
    try:
        r = requests.get(f"{SIMPLY_BASE}/search-results.html",
                         headers=REQUEST_HEADERS, params={"q": title}, timeout=15)
        if r.status_code == 200:
            url = _parse_simply_search(r.text, title)
            if url:
                time.sleep(0.5)
                r2 = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
                if r2.status_code == 200:
                    text = _parse_simply_page(r2.text)
                    if text and len(text.strip()) > 500:
                        save_path.write_text(text, encoding="utf-8")
                        return str(save_path)
    except Exception:
        pass
    return None


# ── ScriptSlug ─────────────────────────────────────────────────────────────

def _parse_slug_page(html: str) -> "str | None":
    soup = BeautifulSoup(html, "html.parser")
    for tag, attrs in [
        ("div", {"class": "script-content"}),
        ("div", {"class": "screenplay"}),
        ("div", {"class": "script"}),
        ("div", {"id":    "script"}),
        ("pre", {}),
    ]:
        elem = soup.find(tag, attrs) if attrs else soup.find(tag)
        if elem and len(elem.get_text()) > 1000:
            return elem.get_text()
    biggest = max((pre.get_text() for pre in soup.find_all("pre")), key=len, default="")
    return biggest if len(biggest) > 2000 else None


def fetch_scriptslug(title: str, tmdb_id: int) -> "str | None":
    """Fetches a script from ScriptSlug via search."""
    save_path   = SCRIPT_DIR / f"{tmdb_id}.txt"
    if save_path.exists():
        return str(save_path)
    title_lower = title.lower().strip()
    try:
        r = requests.get(f"{SLUG_BASE}/results",
                         headers=REQUEST_HEADERS, params={"search": title}, timeout=15)
        if r.status_code != 200:
            return None
        soup       = BeautifulSoup(r.text, "html.parser")
        script_url = None
        for a in soup.find_all("a", href=True):
            href      = a["href"]
            link_text = a.get_text(strip=True).lower()
            if "/script/" not in href and "/scripts/" not in href:
                continue
            clean = re.sub(r"\b(script|screenplay)\b", "", link_text, flags=re.IGNORECASE).strip()
            if clean == title_lower or title_lower in clean or clean in title_lower:
                script_url = href if href.startswith("http") else SLUG_BASE + href
                break
        if not script_url:
            return None
        time.sleep(0.5)
        r2 = requests.get(script_url, headers=REQUEST_HEADERS, timeout=15)
        if r2.status_code != 200:
            return None
        text = _parse_slug_page(r2.text)
        if text and len(text.strip()) > 500:
            save_path.write_text(text, encoding="utf-8")
            return str(save_path)
    except Exception:
        pass
    return None


# ── Alexander Street ───────────────────────────────────────────────────────

def _parse_astreet_page(html: str) -> "str | None":
    soup = BeautifulSoup(html, "html.parser")
    for tag, attrs in [
        ("div",     {"class": "document-text"}),
        ("div",     {"class": "content-text"}),
        ("div",     {"class": "fulltext"}),
        ("section", {"class": "document"}),
        ("article", {}),
        ("main",    {}),
    ]:
        elem = soup.find(tag, attrs) if attrs else soup.find(tag)
        if elem and len(elem.get_text(separator="\n")) > 1000:
            return elem.get_text(separator="\n")
    return None


def _parse_astreet_search(html: str, title: str) -> "str | None":
    soup        = BeautifulSoup(html, "html.parser")
    title_lower = title.lower().strip()
    for a in soup.find_all("a", href=True):
        href      = a["href"]
        link_text = a.get_text(strip=True).lower()
        if not any(x in href for x in ["/view/", "/document/", "afso"]):
            continue
        clean = re.sub(r"\b(script|screenplay|film|the)\b", "", link_text, flags=re.IGNORECASE).strip()
        if clean == title_lower or title_lower in clean or clean in title_lower:
            return href if href.startswith("http") else ASTREET_BASE + href
    return None


def fetch_alexanderstreet(title: str, tmdb_id: int, year: int = None) -> "str | None":
    """Fetches a script from Alexander Street AFSO via keyword search."""
    save_path = SCRIPT_DIR / f"{tmdb_id}.txt"
    if save_path.exists():
        return str(save_path)
    try:
        r = requests.get(
            f"{ASTREET_BASE}/afso",
            headers={**REQUEST_HEADERS, "Accept": "text/html,application/xhtml+xml"},
            params={"q": title, "searchtype": "kw"},
            timeout=20,
        )
        if r.status_code != 200:
            return None
        doc_url = _parse_astreet_search(r.text, title)
        if not doc_url:
            return None
        time.sleep(0.5)
        r2 = requests.get(doc_url, headers=REQUEST_HEADERS, timeout=20)
        if r2.status_code != 200:
            return None
        text = _parse_astreet_page(r2.text)
        if text and len(text.strip()) > 500:
            save_path.write_text(text, encoding="utf-8")
            return str(save_path)
    except Exception:
        pass
    return None


# ── Multi-source orchestration ──────────────────────────────────────────────

def _multi_source_worker(record: dict) -> dict:
    """Tries IMSDb -> SimplyScripts -> ScriptSlug -> Alexander Street in order."""
    if _file_ok(record.get("script_file")):
        return record
    title = record["title"]
    mid   = record["tmdb_id"]
    year  = record.get("release_year")
    for fetcher, label in [
        (lambda: fetch_imsdb_script(title, mid),          "IMSDb"),
        (lambda: fetch_simplyscripts(title, mid),         "SimplyScripts"),
        (lambda: fetch_scriptslug(title, mid),            "ScriptSlug"),
        (lambda: fetch_alexanderstreet(title, mid, year), "Alexander Street"),
    ]:
        result = fetcher()
        if result:
            record["script_file"] = result
            print(f"    [SCRIPT FOUND via {label}] {title} ({year})")
            return record
        time.sleep(0.4)
    return record


def enrich_scripts_multi_source(df: pd.DataFrame) -> pd.DataFrame:
    """
    Scrapes scripts for movies with poster + ROI but no script.
    Only targets movies that could reach movies_clean.csv.
    """
    has_poster = df["poster_file"].apply(_file_ok)
    has_roi    = ~df["roi"].apply(_is_nan)
    has_script = df["script_file"].apply(_file_ok)
    to_enrich  = df[has_poster & has_roi & ~has_script].to_dict("records")
    skip       = df[~(has_poster & has_roi & ~has_script)].to_dict("records")
    print(f"  Candidates (poster + ROI, no script): {len(to_enrich):,}")
    print(f"  Skipping (can't reach clean set):     {len(skip):,}")
    if not to_enrich:
        return df
    enriched = []
    success = fail = 0
    with ThreadPoolExecutor(max_workers=IMSDB_WORKERS) as executor:
        futures = {executor.submit(_multi_source_worker, rec): rec for rec in to_enrich}
        for future in as_completed(futures):
            result   = future.result()
            enriched.append(result)
            found     = _file_ok(result.get("script_file"))
            success  += found
            fail     += not found
            done      = success + fail
            if done % 50 == 0:
                pct = done / len(to_enrich) * 100
                print(f"  Multi-source: {done:,}/{len(to_enrich):,} ({pct:.1f}%)  "
                      f"|  found: {success:,}  |  not found: {fail:,}")
    print(f"\n  Newly found: {success:,}  |  Still missing: {fail:,}")
    df_updated = pd.DataFrame(skip + enriched)
    df_updated.to_csv(BASE_DIR / "movies_raw.csv", index=False)
    print(f"  Updated movies_raw.csv ({len(df_updated):,} rows)")
    return df_updated

## Box Office Mojo ROI Enrichment

Uses `imdb_id` for direct BOM URL lookups (no search step).
Skips movies that already have ROI via `_is_nan()` — not `is not None`,
which incorrectly returns `True` for `float NaN`.

In [ ]:
def _parse_bom_money(text: str) -> "float | None":
    if not text:
        return None
    text = text.strip().replace(",", "").replace("$", "").replace(" ", "")
    try:
        if text.endswith("B"):
            return float(text[:-1]) * 1_000_000_000
        if text.endswith("M"):
            return float(text[:-1]) * 1_000_000
        v = float(text)
        return v if v > 0 else None
    except ValueError:
        return None


def fetch_bom_financials(imdb_id: str, release_year: int) -> dict:
    """
    Scrapes a BOM title page and returns updated financial fields.
    Returns an empty dict if the page is unavailable or yields no data.
    """
    if not imdb_id or _is_nan(imdb_id) or not str(imdb_id).startswith("tt"):
        return {}
    url = f"{BOM_BASE}/title/{imdb_id}/"
    try:
        r = requests.get(url, headers=BOM_HEADERS, timeout=15)
        if r.status_code == 404:
            return {}
        if r.status_code == 429:
            time.sleep(10)
            r = requests.get(url, headers=BOM_HEADERS, timeout=15)
        if r.status_code != 200:
            return {}
        soup = BeautifulSoup(r.text, "html.parser")
        domestic_gross = worldwide_gross = budget = None
        # Primary: BOM summary grid
        for cell in soup.select("div.mojo-performance-summary-table > div"):
            spans = cell.find_all("span")
            if len(spans) < 2:
                continue
            label = spans[0].get_text(strip=True).lower()
            value = _parse_bom_money(spans[-1].get_text(strip=True))
            if value is None:
                continue
            if "domestic"  in label: domestic_gross  = value
            elif "worldwide" in label: worldwide_gross = value
            elif "budget"    in label: budget          = value
        # Fallback: adjacent span pairs
        if not domestic_gross and not worldwide_gross:
            spans = soup.find_all("span")
            for i in range(len(spans) - 1):
                label = spans[i].get_text(strip=True).lower()
                value = _parse_bom_money(spans[i + 1].get_text(strip=True))
                if value is None:
                    continue
                if "domestic"  in label and domestic_gross  is None: domestic_gross  = value
                elif "worldwide" in label and worldwide_gross is None: worldwide_gross = value
                elif "budget"    in label and budget          is None: budget          = value
        # Fallback: table rows
        if not domestic_gross and not worldwide_gross:
            for row in soup.find_all("tr"):
                cells = row.find_all("td")
                if len(cells) < 2:
                    continue
                label = cells[0].get_text(strip=True).lower()
                value = _parse_bom_money(cells[-1].get_text(strip=True))
                if value is None:
                    continue
                if "domestic"  in label: domestic_gross  = value
                elif "worldwide" in label: worldwide_gross = value
                elif "budget"    in label: budget          = value
        gross = worldwide_gross or domestic_gross
        if not gross:
            return {}
        adj_gross, adj_budget, roi = compute_roi(gross, budget or 0, release_year)
        result = {
            "gross_raw":      round(gross, 2),
            "gross_2025":     adj_gross,
            "cpi_gross_year": CPI.get(release_year),
        }
        if budget:
            result["budget_raw"]      = round(budget, 2)
            result["budget_2025"]     = adj_budget
            result["cpi_budget_year"] = CPI.get(release_year - 1)
        if roi is not None:
            result["roi"]     = roi
            result["log_roi"] = round(float(np.log1p(roi)), 6) if roi > -1 else None
        return result
    except Exception:
        return {}


def _bom_worker(record: dict) -> dict:
    if not _is_nan(record.get("roi")):
        return record
    imdb_id      = record.get("imdb_id")
    release_year = record.get("release_year")
    if _is_nan(imdb_id) or _is_nan(release_year):
        return record
    financials = fetch_bom_financials(str(imdb_id), int(release_year))
    for key, val in financials.items():
        existing = record.get(key)
        if _is_nan(existing) or existing == 0:
            record[key] = val
    return record


def enrich_roi_from_bom(df: pd.DataFrame) -> pd.DataFrame:
    """Fetches budget/gross from BOM for movies missing ROI. Saves updated movies_raw.csv."""
    missing_roi = df["roi"].apply(_is_nan)
    to_enrich   = df[missing_roi].to_dict("records")
    already_ok  = df[~missing_roi].to_dict("records")
    print(f"  Movies already with ROI:       {len(already_ok):,}")
    print(f"  Movies missing ROI (BOM pass): {len(to_enrich):,}")
    if not to_enrich:
        return df
    enriched = []
    gained = failed = 0
    with ThreadPoolExecutor(max_workers=TMDB_WORKERS) as executor:
        futures = {executor.submit(_bom_worker, rec): rec for rec in to_enrich}
        for future in as_completed(futures):
            result  = future.result()
            enriched.append(result)
            ok       = not _is_nan(result.get("roi"))
            gained  += ok
            failed  += not ok
            done     = gained + failed
            if done % 500 == 0:
                pct = done / len(to_enrich) * 100
                print(f"  BOM: {done:,}/{len(to_enrich):,} ({pct:.1f}%)  "
                      f"|  ROI gained: {gained:,}  |  still missing: {failed:,}")
    print(f"\n  ROI newly computable: {gained:,}  |  Still missing: {failed:,}")
    df_updated = pd.DataFrame(already_ok + enriched)
    df_updated.to_csv(BASE_DIR / "movies_raw.csv", index=False)
    print(f"  Updated movies_raw.csv ({len(df_updated):,} rows)")
    return df_updated

## Checkpoint Helpers & Pipeline Workers

In [ ]:
def save_checkpoint(records: list, n: int):
    df = pd.DataFrame(records)
    df.to_csv(CKPT_DIR / f"checkpoint_{n:06d}.csv",   index=False)
    df.to_csv(BASE_DIR  / "movies_raw_partial.csv",    index=False)
    print(f"\n  CHECKPOINT -- {n:,} movies | "
          f"posters: {df['poster_file'].notna().sum():,} | "
          f"scripts: {df['script_file'].notna().sum():,} | "
          f"ROI: {df['roi'].notna().sum():,}")


def load_checkpoint():
    partial_path = BASE_DIR / "movies_raw_partial.csv"
    if partial_path.exists():
        df       = pd.read_csv(partial_path)
        records  = df.to_dict("records")
        done_ids = set(df["tmdb_id"].tolist())
        print(f"  Found checkpoint: {len(records):,} records")
        return records, done_ids
    print("  No checkpoint -- starting fresh")
    return [], set()


def fetch_details_worker(tmdb_id: int) -> "dict | None":
    """Fetches and flattens TMDB movie details into a record dict. Returns None for adult/invalid."""
    details = get_movie_details(tmdb_id)
    if not details or details.get("adult"):
        return None
    release_date = details.get("release_date", "")
    release_year = int(release_date[:4]) if release_date else None
    if not release_year:
        return None
    gross  = details.get("revenue", 0) or 0
    budget = details.get("budget",  0) or 0
    adj_gross, adj_budget, roi = compute_roi(gross, budget, release_year)
    log_roi   = float(np.log1p(roi)) if roi is not None and roi > -1 else None
    genres    = [g["name"] for g in details.get("genres", [])]
    companies = [c["name"] for c in details.get("production_companies", [])]
    cert      = get_us_certification(details.get("release_dates", {}))
    return {
        "tmdb_id":           tmdb_id,
        "imdb_id":           details.get("imdb_id"),
        "title":             details.get("title", ""),
        "release_year":      release_year,
        "release_date":      release_date,
        "budget_raw":        budget if budget > 0 else None,
        "gross_raw":         gross  if gross  > 0 else None,
        "budget_2025":       adj_budget,
        "gross_2025":        adj_gross,
        "cpi_gross_year":    CPI.get(release_year),
        "cpi_budget_year":   CPI.get(release_year - 1),
        "roi":               roi,
        "log_roi":           log_roi,
        "runtime_min":       details.get("runtime"),
        "genres":            "|".join(genres),
        "certification":     cert,
        "original_language": details.get("original_language"),
        "popularity":        details.get("popularity"),
        "vote_average":      details.get("vote_average"),
        "vote_count":        details.get("vote_count"),
        "production_cos":    "|".join(companies),
        "tagline":           details.get("tagline"),
        "overview":          details.get("overview"),
        "poster_path":       details.get("poster_path"),
        "poster_file":       None,
        "script_file":       None,
        "poster_synthetic":  False,
    }


def download_poster_worker(record: dict) -> dict:
    record["poster_file"] = download_poster(record["tmdb_id"], record.get("poster_path"))
    return record


def fetch_script_worker(record: dict) -> dict:
    local_script          = fetch_imsdb_script(record["title"], record["tmdb_id"])
    record["script_file"] = local_script
    if local_script:
        print(f"    [SCRIPT FOUND] {record['title']} ({record['release_year']})")
    return record

## Main Pipeline — `build_dataset`

In [ ]:
def build_dataset() -> pd.DataFrame:
    """
    Full parallel pipeline: discover IDs -> fetch details -> download posters
    -> scrape scripts -> save movies_raw.csv.
    Resumes from movies_raw_partial.csv if it exists.
    """
    print("=" * 60)
    print("  STEP 1 -- Check for existing checkpoint")
    print("=" * 60)
    records, done_ids = load_checkpoint()
    print(f"  Total collected so far: {len(records):,}\n")

    print("=" * 60)
    print(f"  STEP 2 -- Discover movie IDs ({START_YEAR}-{END_YEAR})")
    print("=" * 60)
    all_ids = []
    grand_total = 0
    for year in range(START_YEAR, END_YEAR + 1):
        print(f"\n  Discovering {year}...")
        ids = discover_movie_ids(year)
        all_ids.extend(ids)
        grand_total += len(ids)
        print(f"  {year}: {len(ids):,} movies  (running total: {grand_total:,})")
    new_ids = [i for i in all_ids if i not in done_ids]
    print(f"\n  Total: {grand_total:,}  |  Already done: {len(done_ids):,}  |  New: {len(new_ids):,}")

    print("\n" + "=" * 60)
    print(f"  STEP 3 -- Fetch movie details ({TMDB_WORKERS} threads)")
    print("=" * 60)
    raw_records = []
    fetched = skipped = 0
    with ThreadPoolExecutor(max_workers=TMDB_WORKERS) as executor:
        futures = {executor.submit(fetch_details_worker, mid): mid for mid in new_ids}
        for future in as_completed(futures):
            result   = future.result()
            fetched += 1
            if result is None:
                skipped += 1
            else:
                raw_records.append(result)
            if fetched % 200 == 0:
                pct = fetched / len(new_ids) * 100
                print(f"  Details: {fetched:,}/{len(new_ids):,} ({pct:.1f}%)  "
                      f"kept: {len(raw_records):,}  skipped: {skipped:,}")
    print(f"  Kept: {len(raw_records):,}  |  Skipped: {skipped:,}")

    print("\n" + "=" * 60)
    print(f"  STEP 4 -- Download posters ({POSTER_WORKERS} threads)")
    print("=" * 60)
    poster_records = []
    p_success = p_fail = 0
    with ThreadPoolExecutor(max_workers=POSTER_WORKERS) as executor:
        futures = {executor.submit(download_poster_worker, rec): rec for rec in raw_records}
        for future in as_completed(futures):
            result    = future.result()
            poster_records.append(result)
            ok         = bool(result.get("poster_file"))
            p_success += ok
            p_fail    += not ok
            done       = p_success + p_fail
            if done % 200 == 0:
                pct = done / len(raw_records) * 100
                print(f"  Posters: {done:,}/{len(raw_records):,} ({pct:.1f}%)  "
                      f"saved: {p_success:,}  missing: {p_fail:,}")
    print(f"  Downloaded: {p_success:,}  |  Missing: {p_fail:,}")

    print("\n" + "=" * 60)
    print(f"  STEP 5 -- Scrape scripts from IMSDb ({IMSDB_WORKERS} threads)")
    print("=" * 60)
    final_records = []
    s_success = s_fail = 0
    with ThreadPoolExecutor(max_workers=IMSDB_WORKERS) as executor:
        futures = {executor.submit(fetch_script_worker, rec): rec for rec in poster_records}
        for future in as_completed(futures):
            result    = future.result()
            final_records.append(result)
            ok         = bool(result.get("script_file"))
            s_success += ok
            s_fail    += not ok
            done       = s_success + s_fail
            if done % 100 == 0:
                pct = done / len(poster_records) * 100
                print(f"  Scripts: {done:,}/{len(poster_records):,} ({pct:.1f}%)  "
                      f"found: {s_success:,}  not found: {s_fail:,}")
    print(f"  Found: {s_success:,}  |  Not found: {s_fail:,}")

    print("\n" + "=" * 60)
    print("  STEP 6 -- Merge and save")
    print("=" * 60)
    for rec in final_records:
        rec.pop("poster_path", None)
    all_records = records + final_records
    if final_records:
        save_checkpoint(all_records, len(all_records))
    df_raw = pd.DataFrame(all_records)
    df_raw.to_csv(BASE_DIR / "movies_raw.csv", index=False)
    print(f"  Saved movies_raw.csv ({len(df_raw):,} rows)")
    partial = BASE_DIR / "movies_raw_partial.csv"
    if partial.exists():
        partial.unlink()
    return df_raw

## Dataset Cleaning & Splits

- **`movies_poster_roi.csv`** — poster (real or synthetic) + ROI.
- **`movies_clean.csv`** — poster + ROI + script; richest dataset.
- Splits are 70/30 stratified by `release_year`. Years with only 1 movie go entirely to train.

In [ ]:
FINAL_COLS = [
    "title", "release_year", "poster_file", "poster_synthetic",
    "script_file", "gross_raw", "budget_raw", "gross_2025", "budget_2025",
    "cpi_gross_year", "cpi_budget_year", "roi", "log_roi",
    "tmdb_id", "imdb_id", "release_date", "runtime_min",
    "genres", "certification", "original_language",
    "popularity", "vote_average", "vote_count",
    "production_cos", "tagline", "overview",
]


def _select_final_cols(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in FINAL_COLS if c in df.columns]
    return df[cols].reset_index(drop=True)


def make_poster_roi_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Filters to movies with poster on disk (real or synthetic) and computable ROI."""
    n  = len(df)
    df = df[df["poster_file"].apply(_file_ok)]
    print(f"  After poster filter: {len(df):,}  (removed {n - len(df):,})")
    n2 = len(df)
    df = df[~df["roi"].apply(_is_nan)]
    print(f"  After ROI filter:    {len(df):,}  (removed {n2 - len(df):,})")
    return _select_final_cols(df)


def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Filters to movies with poster, script, and computable ROI."""
    n  = len(df)
    df = df[df["poster_file"].apply(_file_ok)]
    print(f"  After poster filter: {len(df):,}  (removed {n - len(df):,})")
    n2 = len(df)
    df = df[df["script_file"].apply(_file_ok)]
    print(f"  After script filter: {len(df):,}  (removed {n2 - len(df):,})")
    n3 = len(df)
    df = df[~df["roi"].apply(_is_nan)]
    print(f"  After ROI filter:    {len(df):,}  (removed {n3 - len(df):,})")
    return _select_final_cols(df)


def make_splits(df: pd.DataFrame):
    """70/30 train/test split stratified by release_year. Singleton years go entirely to train."""
    year_counts = df["release_year"].value_counts()
    rare_years  = year_counts[year_counts < 2].index.tolist()
    if rare_years:
        print(f"  Note: {len(rare_years)} year(s) with <2 movies -> all in train: {sorted(rare_years)}")
    df_rare         = df[df["release_year"].isin(rare_years)]
    df_stratifiable = df[~df["release_year"].isin(rare_years)]
    train_df, test_df = train_test_split(
        df_stratifiable,
        test_size=0.30,
        stratify=df_stratifiable["release_year"],
        random_state=42,
    )
    train_df = pd.concat([train_df, df_rare], ignore_index=True)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

## Script File Matching (Local Directory)

Matches a local directory of `.txt` script files to movies in `movies_raw.csv` by title.

**Pass 1** — normalized lowercase title match.
**Pass 2** — manual title map + CamelCase splitting for filenames that were concatenated.

Movies whose scripts remain unresolved become stub rows in `extra_movies_raw.csv`
for TMDB enrichment.

In [ ]:
MANUAL_TITLE_MAP = {
    '500 Days of Summer':                           '(500) Days of Summer',
    'TheSelmaLouise':                               'Thelma & Louise',
    'thetragedyofmacbeth':                          'The Tragedy of Macbeth',
    'Theat':                                        'The Heat',
    'threethousandyearsoflonging':                  'Three Thousand Years of Longing',
    'TheUnitedStatesVSBillieHoliday':               'The United States vs. Billie Holiday',
    'triangleofsadness':                            'Triangle of Sadness',
    'Aliens Vs Predato':                            'Aliens vs. Predator',
    '48 Hou':                                       '48 Hrs.',
    'Avenge':                                       'The Avengers',
    'Kramer versus Krame':                          'Kramer vs. Kramer',
    'F1':                                           'F1',
    'Romy and Michelles High School Reunion':       "Romy and Michele's High School Reunion",
    # Truncated filenames
    'Big Lebowsk':                                  'The Big Lebowski',
    'Mean Stree':                                   'Mean Streets',
    'Taxi Drive':                                   'Taxi Driver',
    'True Lie':                                     'True Lies',
    'Three Days of the Condo':                      'Three Days of the Condor',
    'Traff':                                        'Traffic',
    '9th Gate':                                     'The Ninth Gate',
    'Gandh':                                        'Gandhi',
    'Bamboozeled':                                  'Bamboozled',
    'Mr Brook':                                     'Mr. Brooks',
    'Last of the Mochican':                         'The Last of the Mohicans',
    'At First Site':                                'At First Sight',
    'Bound First Draf':                             'Bound',
    'Clerk':                                        'Clerks',
    'The God Fathe':                                'The Godfather',
    'Hacke':                                        'Hackers',
    'As Good As It Ge':                             'As Good as It Gets',
    'Pirscilla Queen of the Dese':                  'The Adventures of Priscilla, Queen of the Desert',
    'Schindlers L':                                 "Schindler's List",
    'Tomorrow Never Die':                           'Tomorrow Never Dies',
    'Space Ball':                                   'Spaceballs',
    'Ghostbuste':                                   'Ghostbusters',
    'Basquia':                                      'Basquiat',
    'Glengarry Glen Ro':                            'Glengarry Glen Ross',
    'Gladiato':                                     'Gladiator',
    'Messenge':                                     'The Messenger',
    'TomRaider':                                    'Tomb Raider',
    'Braveheart S':                                 'Braveheart',
    'Braveheart Tran':                              'Braveheart',
    'Glengarry Glen Gross':                         'Glengarry Glen Ross',
    # Typos
    'Malcom X':                                     'Malcolm X',
    'Emilia Prez':                                  'Emilia P\u00e9rez',
    'Amlie':                                        'Am\u00e9lie',
    'Tinker Tailor Solider Spy':                    'Tinker Tailor Soldier Spy',
    'TinkerTailorSoldierSpy':                       'Tinker Tailor Soldier Spy',
    'Peasantville':                                 'Pleasantville',
    'ForestGump':                                   'Forrest Gump',
    'Romeo Julie':                                  'Romeo + Juliet',
    # "The" moved to end
    'Favourite The':                                'The Favourite',
    'Crown The':                                    'The Crown',
    'Good Wife The':                                'The Good Wife',
    'Good Place The':                               'The Good Place',
    'Kids Are Alright The':                         'The Kids Are All Right',
    'Last Ship The':                                'The Last Ship',
    'Prom The':                                     'The Prom',
    'Place Beyond the Pine The':                    'The Place Beyond the Pines',
    'Horse Whisperer The':                          'The Horse Whisperer',
    # Missing apostrophe
    "Childs Play":                                  "Child's Play",
    "Sharkys Machine":                              "Sharky's Machine",
    "Charlie Wilsons War":                          "Charlie Wilson's War",
    "Nick and Norahs Infinite Playlist":            "Nick and Norah's Infinite Playlist",
    "The Hitmans Bodyguard":                        "The Hitman's Bodyguard",
    "Mary Shelleys Frankenstein":                   "Mary Shelley's Frankenstein",
    "Bram Stokers Dracula":                         "Bram Stoker's Dracula",
    "National Lampoons Vacation":                   "National Lampoon's Vacation",
    "National Lampoons Animal House":               "National Lampoon's Animal House",
    "National Lampoons Christmas Vacation":         "National Lampoon's Christmas Vacation",
    "National Lampoons European Vacation":          "National Lampoon's European Vacation",
    "Harry Potter and the Sorcerers Stone":         "Harry Potter and the Sorcerer's Stone",
    "Hangin with the Homeboys":                     "Hangin' with the Homeboys",
    "A Knights Tale":                               "A Knight's Tale",
    "Frankensteins Army":                           "Frankenstein's Army",
    "Guillermo del Toros Pinocchio":                "Guillermo del Toro's Pinocchio",
    "Bridget Joness Diary":                         "Bridget Jones's Diary",
    "I Dont Feel at Home in This World Anymore":    "I Don't Feel at Home in This World Anymore",
    "The Devils Rejects":                           "The Devil's Rejects",
    "Were No Angels":                               "We're No Angels",
    "To All the Boys Ive Loved Before":             "To All the Boys I've Loved Before",
    "I Think Were Alone Now":                       "I Think We're Alone Now",
    "Get Rich or Die Tryin":                        "Get Rich or Die Tryin'",
    "Dont Look Up":                                 "Don't Look Up",
    "Singin in the Rain":                           "Singin' in the Rain",
    "Trick r Treat":                                "Trick 'r Treat",
    "Talkin Dirty After Dark":                      "Talkin' Dirty After Dark",
    "Carlitos Way":                                 "Carlito's Way",
    "Bill Teds Bogus Journey":                      "Bill & Ted's Bogus Journey",
    "Bill Teds Excellent Adventure":                "Bill & Ted's Excellent Adventure",
    "Thelma Louise":                                "Thelma & Louise",
    # Missing colon (subtitle)
    'Mission Impossible Fallout':                   'Mission: Impossible \u2013 Fallout',
    'MIssion Impossible Rogue Nation':              'Mission: Impossible \u2013 Rogue Nation',
    'Mission Impossible III':                       'Mission: Impossible III',
    'The Hobbit An Unexpected Journey':             'The Hobbit: An Unexpected Journey',
    'The Hobbit The Desolation of Smaug':           'The Hobbit: The Desolation of Smaug',
    'Glass Onion A Knives Out Mystery':             'Glass Onion: A Knives Out Mystery',
    'Top Gun Maverick':                             'Top Gun: Maverick',
    'Anchorman The Legend of Ron Burgundy':         'Anchorman: The Legend of Ron Burgundy',
    'Captain America The First Avenger':            'Captain America: The First Avenger',
    'Mad Max Fury Road':                            'Mad Max: Fury Road',
    'Walk Hard The Dewey Cox Story':                'Walk Hard: The Dewey Cox Story',
    'Zombieland Double Tap':                        'Zombieland: Double Tap',
    'Twin Peaks Fire Walk with Me':                 'Twin Peaks: Fire Walk with Me',
    'The Hunger Games Catching Fire':               'The Hunger Games: Catching Fire',
    'Harry Potter and the Deathly Hallows Part 2':  'Harry Potter and the Deathly Hallows: Part 2',
    'Sicario Day of the Soldado':                   'Sicario: Day of the Soldado',
    'D3 The Mighty Ducks':                          'D3: The Mighty Ducks',
    'D2 The Mighty Ducks':                          'D2: The Mighty Ducks',
    'Abraham Lincoln Vampire Hunter':               'Abraham Lincoln: Vampire Hunter',
    'Alien Covenant':                               'Alien: Covenant',
    'Popstar Never Stop Never Stopping':            'Popstar: Never Stop Never Stopping',
    'X2 X Men United':                             'X2: X-Men United',
    'Team America World Police':                    'Team America: World Police',
    'Airplane II The Sequel':                       'Airplane II: The Sequel',
    'Gremlins 2 The New Batch':                     'Gremlins 2: The New Batch',
    'AVP Alien vs Predator':                        'AVP: Alien vs. Predator',
    'Zathura A Space Adventure':                    'Zathura: A Space Adventure',
    'The Boondock Saints II All Saints Day':        'The Boondock Saints II: All Saints Day',
    'Dominion Prequel to The Exorcist':             'Dominion: Prequel to the Exorcist',
    'The Lost World Jurassic Park':                 'The Lost World: Jurassic Park',
    'Highlander II The Quickening':                 'Highlander II: The Quickening',
    'Batman Mask of the Phantasm':                  'Batman: Mask of the Phantasm',
    'The Mummy Tomb of the Dragon Emperor':         'The Mummy: Tomb of the Dragon Emperor',
    'Exorcist II The Heretic':                      'Exorcist II: The Heretic',
    'Police Academy 2 Their First Assignment':      'Police Academy 2: Their First Assignment',
    'Police Academy 3 Back in Training':            'Police Academy 3: Back in Training',
    'Home Alone 2 Lost in New York':                'Home Alone 2: Lost in New York',
    'Star Trek III The Search for Spock':           'Star Trek III: The Search for Spock',
    'Star Trek IV The Voyage Home':                 'Star Trek IV: The Voyage Home',
    'The Naked Gun From the Files of Police Squad': 'The Naked Gun: From the Files of Police Squad!',
    '2010 The Year We Make Contact':                '2010: The Year We Make Contact',
    'Friday the 13th Part VI Jason Lives':          'Friday the 13th Part VI: Jason Lives',
    'Halloween 4 The Return of Michael Myers':      'Halloween 4: The Return of Michael Myers',
    'Halloween H20 20 Years Later':                 'Halloween H20: 20 Years Later',
    'Terminator 2 Judgment Day':                    'Terminator 2: Judgment Day',
    'A Nightmare on Elm Street The Dream Child':    'A Nightmare on Elm Street 5: The Dream Child',
    'A Nightmare On Elm Street 3 Dream Warriors':   'A Nightmare on Elm Street 3: Dream Warriors',
    # Missing & / special punctuation
    'Love Other Drugs':                             'Love & Other Drugs',
    'Malcolm Marie':                                'Malcolm & Marie',
    'Mr Mrs Smith':                                 'Mr. & Mrs. Smith',
    'Hustle Flow':                                  'Hustle & Flow',
    'Willy Wonka the Chocolate Factory':            'Willy Wonka & the Chocolate Factory',
    'Monsters Inc':                                 'Monsters, Inc.',
    'Extremely Wicked Shockingly Evil and Vile':    'Extremely Wicked, Shockingly Evil and Vile',
    'I Love You Man':                               'I Love You, Man',
    'Love Simon':                                   'Love, Simon',
    'Planes Trains Automobiles':                    'Planes, Trains and Automobiles',
    'Sex Lies and Videotape':                       'sex, lies, and videotape',
    'Frost Nixon':                                  'Frost/Nixon',
    'Jo Jo Dancer Your Life is Calling':            'Jo Jo Dancer, Your Life Is Calling',
    'When Harry Met Sally':                         'When Harry Met Sally...',
    'Mars Attacks':                                 'Mars Attacks!',
    'Oceans Eight':                                 "Ocean's Eight",
    'Dr Strangelove or How I Learned to Stop Worrying and Love the Bomb':
                                                    'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb',
    # Hyphen / dot missing
    'ET the Extra Terrestrial':                     'E.T. the Extra-Terrestrial',
    'Dr No':                                        'Dr. No',
    'Mr Mom':                                       'Mr. Mom',
    'Mr Nobody':                                    'Mr. Nobody',
    'K Pax':                                        'K-PAX',
    'The X Files':                                  'The X-Files',
    'Spider Man No Way Home':                       'Spider-Man: No Way Home',
    'X Men Days of Future Past':                    'X-Men: Days of Future Past',
    # Misc
    'Tango Cash':                                   'Tango & Cash',
    'A Soldiers Story':                             "A Soldier's Story",
    'Another 48 Hrs':                               '48 Hrs.',
    'The Mitchells vs the Machines':                'The Mitchells vs. the Machines',
    'Weird The Al Yankovic Story':                  'Weird: The Al Yankovic Story',
    'Whitney Houston I Wanna Dance with Somebody':  'Whitney Houston: I Wanna Dance with Somebody',
}


def _split_camel_case(title: str) -> str:
    return re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', title)


def match_local_scripts(df: pd.DataFrame, scripts_path: str):
    """
    Pass 1: matches .txt files in scripts_path to movies by normalized title.
    Returns (updated_df, script_df) where script_df indexes all found script files.
    """
    script_files_data = []
    for filename in os.listdir(scripts_path):
        if filename.endswith('.txt'):
            full_path   = os.path.join(scripts_path, filename)
            movie_title = filename.replace('-', ' ').replace('.txt', '').strip()
            script_files_data.append({
                'extracted_title':      movie_title,
                'script_file_new_path': full_path,
            })
    script_df = pd.DataFrame(script_files_data)
    script_df['normalized_extracted_title'] = script_df['extracted_title'].str.lower()
    df = df.copy()
    df['normalized_title']      = df['title'].str.lower()
    title_to_path               = script_df.set_index('normalized_extracted_title')['script_file_new_path'].to_dict()
    df['script_file_candidate'] = df['normalized_title'].map(title_to_path)
    df['script_file']           = df['script_file_candidate'].fillna(df['script_file'])
    df = df.drop(columns=['normalized_title', 'script_file_candidate'])
    print(f"  Pass 1 matched: {df['script_file'].notna().sum():,} movies")
    return df, script_df


def resolve_unmatched_scripts(df: pd.DataFrame, script_df: pd.DataFrame, manual_title_map: dict):
    """
    Pass 2: resolves scripts not matched in pass 1 using the manual title map
    and CamelCase splitting. Unresolvable scripts become stub rows in
    extra_movies_raw for TMDB enrichment.
    Returns (updated_df, extra_movies_df).
    """
    matched_paths  = set(df.dropna(subset=['script_file'])['script_file'].tolist())
    unmatched_rows = script_df[~script_df['script_file_new_path'].isin(matched_paths)].copy()
    df_title_lower = {t.lower(): t for t in df['title'].dropna().tolist()}
    manual_lower   = {k.lower(): v for k, v in manual_title_map.items()}
    df = df.copy()
    new_matches = 0
    still_unmatched = []
    for _, row in unmatched_rows.iterrows():
        extracted   = row['extracted_title']
        script_path = row['script_file_new_path']
        resolved    = manual_lower.get(extracted.lower())
        if resolved is None:
            split = _split_camel_case(extracted)
            if split.lower() in df_title_lower:
                resolved = split
        if resolved is not None:
            mask = df['title'].str.lower() == resolved.lower()
            if mask.any():
                update_mask = mask & df['script_file'].isna()
                if update_mask.any():
                    df.loc[update_mask, 'script_file'] = script_path
                new_matches += 1
            else:
                still_unmatched.append(extracted)
        else:
            still_unmatched.append(extracted)
    print(f"  Pass 2 new matches:            {new_matches}")
    print(f"  Total movies with script:      {df['script_file'].notna().sum():,}")
    print(f"  Still unmatched script files:  {len(still_unmatched)}")
    extracted_to_path = script_df.set_index('extracted_title')['script_file_new_path'].to_dict()
    extra_rows = []
    for extracted in still_unmatched:
        script_path   = extracted_to_path.get(extracted)
        cleaned_title = manual_lower.get(extracted.lower()) or _split_camel_case(extracted)
        extra_rows.append({'title': cleaned_title, 'script_file': script_path})
    extra_df = pd.concat(
        [pd.DataFrame(columns=df.columns), pd.DataFrame(extra_rows)],
        ignore_index=True,
    ).reindex(columns=df.columns)
    return df, extra_df

## Extra Movies Enrichment

Enriches `extra_movies_raw.csv` (unmatched scripts) with full TMDB metadata + posters,
then runs a BOM pass for ROI. The original `script_file` path is always preserved.

In [ ]:
def enrich_extra_worker(record: dict) -> dict:
    """
    Per-row worker: search TMDB by title -> fetch full details -> download poster.
    Skips if already fully enriched. Never overwrites the original script_file.
    """
    title           = record.get("title")
    original_script = record.get("script_file")
    if not title or _is_nan(title):
        return record
    tmdb_id = record.get("tmdb_id")
    if _is_nan(tmdb_id):
        tmdb_id = search_tmdb_by_title(title, record.get("release_year"))
    if tmdb_id is None:
        return record
    details = fetch_details_worker(int(tmdb_id))
    if details is None:
        return record
    details["script_file"] = original_script
    details = download_poster_worker(details)  # poster_path still present at this point
    details.pop("poster_path", None)
    return details


def enrich_extra_movies(df: pd.DataFrame) -> pd.DataFrame:
    """
    Enriches all rows in extra_movies_raw with TMDB metadata + posters.
    Runs a BOM pass for ROI after the TMDB pass. Saves result to EXTRA_PATH.
    """
    records = df.to_dict("records")
    print("=" * 60)
    print(f"  Enriching {len(records):,} extra movies via TMDB")
    print("=" * 60)
    enriched = []
    found = not_found = 0
    with ThreadPoolExecutor(max_workers=TMDB_WORKERS) as executor:
        futures = {executor.submit(enrich_extra_worker, rec): rec for rec in records}
        for future in as_completed(futures):
            result     = future.result()
            enriched.append(result)
            ok          = not _is_nan(result.get("tmdb_id"))
            found      += ok
            not_found  += not ok
            done        = found + not_found
            if done % 50 == 0:
                pct = done / len(records) * 100
                print(f"  {done:,}/{len(records):,} ({pct:.1f}%)  "
                      f"|  matched: {found:,}  |  unmatched: {not_found:,}")
    print(f"\n  TMDB matched: {found:,}  |  Not found: {not_found:,}")
    df_enriched = pd.DataFrame(enriched)
    for col in df.columns:
        if col not in df_enriched.columns:
            df_enriched[col] = None
    df_enriched = df_enriched.reindex(columns=df.columns)
    print("\n" + "=" * 60)
    print("  BOM enrichment pass for ROI")
    print("=" * 60)
    df_enriched = enrich_roi_from_bom(df_enriched)
    df_enriched.to_csv(EXTRA_PATH, index=False)
    print(f"\n  Saved -> {EXTRA_PATH}  ({len(df_enriched):,} rows)")
    print(f"  Rows with tmdb_id: {df_enriched['tmdb_id'].notna().sum():,}")
    print(f"  Rows with poster:  {df_enriched['poster_file'].apply(_file_ok).sum():,}")
    print(f"  Rows with ROI:     {(~df_enriched['roi'].apply(_is_nan)).sum():,}")
    print(f"  Rows with script:  {df_enriched['script_file'].apply(_file_ok).sum():,}")
    return df_enriched

## Run Pipeline

Set the flags below to control which stages execute, then run this cell.

| Flag | Default | Effect |
|------|---------|--------|
| `RUN_BUILD_DATASET` | `False` | Scrape all TMDB IDs from scratch (skip if `movies_raw.csv` exists) |
| `RUN_BOM_ENRICHMENT` | `False` | Fetch missing ROI from Box Office Mojo |
| `RUN_SCRIPT_ENRICHMENT` | `False` | Scrape additional script sources |
| `BUILD_POSTER_ROI_CSV` | `True` | Produce `movies_poster_roi.csv` + splits |
| `BUILD_CLEAN_CSV` | `True` | Produce `movies_clean.csv` + splits |

In [ ]:
RUN_BUILD_DATASET     = False
RUN_BOM_ENRICHMENT    = False
RUN_SCRIPT_ENRICHMENT = False
BUILD_POSTER_ROI_CSV  = True
BUILD_CLEAN_CSV       = True

# ── Load or build raw data ────────────────────────────────────────────────
if RUN_BUILD_DATASET:
    df_raw = build_dataset()
else:
    df_raw = pd.read_csv(BASE_DIR / "movies_raw.csv")
    print(f"Loaded movies_raw.csv: {len(df_raw):,} rows")
    print(f"  with ROI:    {(~df_raw['roi'].apply(_is_nan)).sum():,}")
    print(f"  with poster: {df_raw['poster_file'].apply(_file_ok).sum():,}")
    print(f"  with script: {df_raw['script_file'].apply(_file_ok).sum():,}")

# ── BOM ROI enrichment ─────────────────────────────────────────────────────
if RUN_BOM_ENRICHMENT:
    print("\n" + "=" * 60)
    print("  ROI enrichment from Box Office Mojo")
    print("=" * 60)
    df_raw = enrich_roi_from_bom(df_raw)

# ── Synthetic posters ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  Synthetic poster generation")
print("=" * 60)
df_raw = fill_missing_posters(df_raw)

# ── Script enrichment ──────────────────────────────────────────────────────
if RUN_SCRIPT_ENRICHMENT:
    print("\n" + "=" * 60)
    print("  Script enrichment (multi-source)")
    print("=" * 60)
    df_raw = enrich_scripts_multi_source(df_raw)

# ── Vote-average dataset ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  Building movies_vote_avg.csv")
print("=" * 60)
df_vote = (
    df_raw
    .loc[df_raw["poster_file"].apply(_file_ok)]
    .loc[df_raw["vote_average"].notna()]
    .loc[df_raw["vote_count"] >= 50]
    .reset_index(drop=True)
)
print(f"  Rows: {len(df_vote):,}")
df_vote.to_csv(BASE_DIR / "movies_vote_avg.csv", index=False)
vote_train, vote_test = make_splits(df_vote)
vote_train.to_csv(BASE_DIR / "movies_vote_avg_train.csv", index=False)
vote_test.to_csv( BASE_DIR / "movies_vote_avg_test.csv",  index=False)
print(f"  Train: {len(vote_train):,}  |  Test: {len(vote_test):,}")

# ── Poster + ROI dataset ───────────────────────────────────────────────────
if BUILD_POSTER_ROI_CSV:
    print("\n" + "=" * 60)
    print("  Building movies_poster_roi.csv")
    print("=" * 60)
    df_poster_roi = make_poster_roi_dataset(df_raw)
    df_poster_roi.to_csv(BASE_DIR / "movies_poster_roi.csv", index=False)
    print(f"  Rows: {len(df_poster_roi):,}")
    pr_train, pr_test = make_splits(df_poster_roi)
    pr_train.to_csv(BASE_DIR / "movies_poster_roi_train.csv", index=False)
    pr_test.to_csv( BASE_DIR / "movies_poster_roi_test.csv",  index=False)
    print(f"  Train: {len(pr_train):,}  |  Test: {len(pr_test):,}")

# ── Full clean dataset (poster + script + ROI) ────────────────────────────
if BUILD_CLEAN_CSV:
    print("\n" + "=" * 60)
    print("  Building movies_clean.csv")
    print("=" * 60)
    df_clean = clean_dataset(df_raw)
    df_clean.to_csv(BASE_DIR / "movies_clean.csv", index=False)
    print(f"  Rows: {len(df_clean):,}")
    if len(df_clean) > 1:
        train_df, test_df = make_splits(df_clean)
        train_df.to_csv(BASE_DIR / "movies_train.csv", index=False)
        test_df.to_csv( BASE_DIR / "movies_test.csv",  index=False)
        print(f"  Train: {len(train_df):,}  |  Test: {len(test_df):,}")
    else:
        print("  [SKIP] Not enough rows to split.")

print("\nDone.")

## Run Local Script Matching

Run after `movies_raw.csv` exists to match a local script directory
and produce `extra_movies_raw.csv`.

In [ ]:
FILE_PATH    = BASE_DIR / "movies_raw.csv"
SCRIPTS_PATH = str(BASE_DIR / "Anyis_Script/Movie-Script-Database/scripts/filtered")
EXTRA_PATH   = BASE_DIR / "extra_movies_raw.csv"

df_raw_files = pd.read_csv(FILE_PATH)
print(f"Loaded {len(df_raw_files):,} rows from movies_raw.csv")

df_raw_files, script_df = match_local_scripts(df_raw_files, SCRIPTS_PATH)
df_raw_files, extra_movies_raw = resolve_unmatched_scripts(
    df_raw_files, script_df, MANUAL_TITLE_MAP
)

df_raw_files.to_csv(FILE_PATH, index=False)
print(f"\nUpdated movies_raw.csv ({len(df_raw_files):,} rows)")
extra_movies_raw.to_csv(EXTRA_PATH, index=False)
print(f"Saved extra_movies_raw.csv ({len(extra_movies_raw):,} rows)")

## Run Extra Movies Enrichment

In [ ]:
extra_movies_raw = pd.read_csv(EXTRA_PATH)
print(f"Loaded {len(extra_movies_raw):,} rows from extra_movies_raw.csv")
extra_movies_raw = enrich_extra_movies(extra_movies_raw)